## Data Extraction and Processing
- From the entire maestro dataset, extract the music of Chopin
- Split the data into training, testing and validation using metadata csv
- Put them in the separate dataset/ directory

In [ ]:
import os
from pathlib import Path
import csv
import shutil
from typing import DefaultDict, Optional

# Directory and file paths used throughout preprocessing
PROJECT_ROOT: Path = Path.cwd()
MAESTRO_DIR: Path = PROJECT_ROOT / "maestro-v3.0.0"
MAESTRO_METADATA_CSV: Path = MAESTRO_DIR / "maestro-v3.0.0.csv"
DATASET_DIR: Path = PROJECT_ROOT / "dataset"

# Composer name as written in MAESTRO metadata
COMPOSER_NAME: str = "Frédéric Chopin"
UNPROCESSED_DIR: Path = DATASET_DIR / f"unprocessed_{COMPOSER_NAME}"
MELODY_DIR: Path = DATASET_DIR / "melodies"
FILTERED_METADATA_CSV: Path = DATASET_DIR / "metadata.csv"

# Ensure output dataset directories exist.
UNPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MELODY_DIR.mkdir(parents=True, exist_ok=True)

copied_rows: list[dict[str, str]] = []
counts: dict[str, int] = {"train": 0, "validation": 0, "test": 0}

# Read MAESTRO metadata and copy only Chopin files into split folders.
with MAESTRO_METADATA_CSV.open("r", encoding="utf-8", newline="") as f:
    reader: csv.DictReader = csv.DictReader(f)
    for row in reader:
        # Keep only Chopin's work
        if row.get("canonical_composer") != COMPOSER_NAME:
            continue

        split_type: str = row["split"]
        canonical_title: str = row["canonical_title"]

        midi_filename_path: Path = Path(row["midi_filename"])
        source_midi: Path = MAESTRO_DIR / midi_filename_path

        target_midi: Path = UNPROCESSED_DIR / split_type / f"{canonical_title}.midi"
        target_midi.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_midi, target_midi)

        out_row: dict[str, str] = dict(row)
        # Remove irrelevant fields.
        for field in ("canonical_composer", "midi_filename", "audio_filename", "year", "duration"):
            out_row.pop(field, None)
        out_row["split"] = split_type
        out_row["midi_path"] = str(target_midi.relative_to(PROJECT_ROOT))
        copied_rows.append(out_row)

        # Update counts
        counts[split_type] += 1

if copied_rows:
    metadata_fields: list[str] = list(copied_rows[0].keys())
    with FILTERED_METADATA_CSV.open("w", encoding="utf-8", newline="") as f:
        writer: csv.DictWriter = csv.DictWriter(f, fieldnames=metadata_fields)
        writer.writeheader()
        writer.writerows(copied_rows)

print(f"Composer name: {COMPOSER_NAME}")
print(f"Copied MIDI files: {len(copied_rows)}")
print(f"Metadata file: {FILTERED_METADATA_CSV}")
for split_name in ["train", "validation", "test"]:
    print(f"  {split_name}: {counts[split_name]} files")

# Melody Extraction from MIDI

- Extract a continuous melodic line from the full piano MIDI file. Usually, the right hand plays the melody and the left hand the harmony. This means we can roughly approximate the higher pitches (rightwards keys) as melodic and lower pitches (leftward keys) as harmonic. The reference "middle key" is C4.

- The key for meaningful data extraction is also keeping important musical context like tempo, time signature, and key signature. Moreover, even preserving some harmonic info is important.

- The musical context is important, otherwise, we could have only the rightmost (highest) pitches being played in order, which would essentially "destroy" the melody.

- Other heuristics include:
    - Keeping pitch changes smooth to avoid unnatural jumps.
    - Preserving the timing and phrasing of the original piece.
    - Using candidate values when there are no right-hand notes.


- It is also important to convert the "multiple parts" into a single sequence of notes represented by "time-series" data for the LSTM training. This is done by music21's "flatten" method.

## The Extraction Process

### Step 1: Parse the MIDI File

Load the MIDI file using music21 and convert it into a `Score` object. This allows us to directly access the properties of the MIDI file such as the notes, chords, timing, and tempo marks.

### Step 2: Collect Note Candidates by Time

Iterate through all the musical events in the piece and group them by their start time (called the **offset** in music21, measured in quarter lengths).

We create a map that looks like this:

``offset (time) → [note1, note2, ...]``


Each position in time might have:
- A single note (easy case)
- Multiple notes from a chord (need to choose one)
- Rest or silence (skip it)

This grouping helps us decide which note is the "melody" at each moment.

### Step 3: Handle Different Note Types

As we go through the flattened stream of events, we encounter two types of pitched elements:

#### Single Notes
If we hit a single note, it's a direct melody candidate. We store both the note object and its MIDI pitch number.

#### Chords
If we hit a chord (multiple notes at the same time), we expand it into individual "pseudo-notes." Each pitch in the chord becomes a separate candidate. This way, instead of throwing away the chord, we can pick the best single note from it to represent that moment in the melody.

### Step 4: Create the Output Melody Part

We create a new empty musical part that will hold our extracted melody. We also copy over important context events from the original score:
- **Tempo marks** (MetronomeMark) — tells us how fast to play
- **Time signatures** (TimeSignature) — tells us the beat structure
- **Key signatures** (KeySignature) — tells us the tonal center

This way, when we export the melody back to MIDI, it will sound right in the original context.

The batch extractor builds output names and passes them into `extract_melody`.
The current naming pattern is:

``dataset/melody/<split>/<input_filename>_melody.midi``

### Step 5: Choose the Best Melodic Note

For each moment in time, we have a list of candidate notes and need to pick one. The algorithm uses two main rules:

1. Start by picking the highest pitch (mentioned above).
2. Minimize the pitch jump from the previous melody note. Big jumps sound odd.
3. If there's a tie, prefer the higher pitch (again, melodies tend higher).
4. Only consider notes in the "right hand" range (>= C4). If no notes are in that range, we consider all candidates.

### Step 6: Handle Gaps and Rests

If there's a time gap between the end of one melody note and the start of the next, we insert a rest. This preserves the phrasing and spacing of the original piece. Without rests, the melody would sound rushed.

## Why This Works

The algorithm makes intuitive musical choices:
1. Prefer notes in the right-hand register (where melodies typically live).
2. The result is a single melodic line that captures the essence of what a pianist would recognize as "the tune" of the piece.

## Key Variables

| Variable        | Meaning |
|-----------------|---------|
| `offset`        | Start time of a note, measured in quarter lengths |
| `quarterLength` | Duration of a note in quarter notes (standard MIDI unit) |
| `midi` (pitch number) | 0-127 scale where 60 = middle C, higher = higher pitch |
| `velocity`      | How hard the note was struck (0-127); we preserve this for dynamics |

In [ ]:
from collections import defaultdict
from music21 import converter, stream, note, chord, meter, tempo, key


def extract_melody(input_filename: str, output_filename: str) -> None:
    # Parse the MIDI into a full score object (all tracks/parts merged in one container).
    score = converter.parse(input_filename)

    # Store note candidates grouped by onset time.
    events_by_offset: DefaultDict[float, list[tuple[note.Note, int]]] = defaultdict(list)

    # Iterate over all pitched musical events after flattening nested parts/measures.
    # flatten().notes returns Note and Chord events in a single time-ordered stream.
    for el in score.flatten().notes:
        if isinstance(el, note.Note):
            # Single notes are already melody candidates; store directly.
            events_by_offset[float(el.offset)].append((el, int(el.pitch.midi)))
        elif isinstance(el, chord.Chord):
            # Chords are expanded into independent pseudo-notes so we can choose one pitch
            # as the melody at that onset (instead of discarding the chord completely).
            for p in el.pitches:
                pseudo: note.Note = note.Note(p)
                # Preserve rhythmic value from the source chord.
                pseudo.quarterLength = float(el.quarterLength)
                # Preserve velocity so exported melody keeps rough dynamic shape.
                pseudo.volume.velocity = el.volume.velocity
                events_by_offset[float(el.offset)].append((pseudo, int(p.midi)))

    # Output part that will contain only one melodic note per onset.
    melody: stream.Part = stream.Part(id="RightHandMelody")

    # Copy global context events (tempo/time/key) from the source score.
    # This keeps timing and musical interpretation consistent when re-exporting.
    for cls in (tempo.MetronomeMark, meter.TimeSignature, key.KeySignature):
        for ctx in score.flatten().getElementsByClass(cls):
            melody.insert(float(ctx.offset), ctx)

    # Track the previous chosen pitch to enforce melodic continuity.
    last_pitch: Optional[int] = None
    # Track end time of last chosen note to insert rests between phrases.
    last_end: float = 0.0

    # Process each onset in chronological order.
    for offset in sorted(events_by_offset.keys()):
        # All notes that start at this exact onset.
        candidates: list[tuple[note.Note, int]] = events_by_offset[offset]

        # Right-hand proxy: prefer pitches >= C4 (MIDI 60).
        # If no such notes exist at this onset, fall back to all candidates.
        right_hand: list[tuple[note.Note, int]] = [c for c in candidates if c[1] >= 60] or candidates

        # Melody choice rule:
        # 1) first onset -> take highest pitch (common melody assumption)
        # 2) later onsets -> minimize pitch jump from previous melody note
        #    and break ties by choosing the higher pitch.
        if last_pitch is None:
            selected_note, selected_pitch = max(right_hand, key=lambda x: x[1])
        else:
            selected_note, selected_pitch = min(
                right_hand,
                key=lambda x: (abs(x[1] - last_pitch), -x[1]),
            )

        onset: float = float(offset)
        # Guard against zero-duration artifacts by enforcing a tiny minimum duration.
        duration: float = max(0.125, float(selected_note.quarterLength))

        # If there is a temporal gap since the previous melody event, insert a rest
        # so the extracted melody preserves phrase spacing.
        if onset > last_end:
            melody.insert(last_end, note.Rest(quarterLength=onset - last_end))

        # Create a fresh note event for output with chosen pitch and preserved duration/velocity.
        out_note: note.Note = note.Note(selected_pitch)
        out_note.quarterLength = duration
        out_note.volume.velocity = selected_note.volume.velocity
        melody.insert(onset, out_note)

        # Update continuity state for the next onset.
        last_pitch = selected_pitch
        last_end = max(last_end, onset + duration)

    melody.write("midi", fp=output_filename)


# Extract melodies from all the dataset
- Convert all Chopin's dataset into melodies.
- Use multithreading to work in parallel.
- Each source file always produces one exported melody MIDI file.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

all_unprocessed_midis: list[Path] = list(UNPROCESSED_DIR.rglob("*.midi"))
jobs: list[tuple[str, str]] = []

for midi_path in all_unprocessed_midis:
    split_name: str = midi_path.relative_to(UNPROCESSED_DIR).parts[0]
    output_midi_path: Path = MELODY_DIR / split_name / f"{midi_path.stem}_melody.midi"
    output_midi_path.parent.mkdir(parents=True, exist_ok=True)
    jobs.append((str(midi_path), str(output_midi_path)))

total: int = len(all_unprocessed_midis)
count: int = 0

with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    futures = {
        pool.submit(extract_melody, input_filename, output_filename): output_filename
        for input_filename, output_filename in jobs
    }
    for future in as_completed(futures):
        future.result()
        count += 1
        processed_file: str = futures[future]
        print(f"({count}/{total}) Completed {Path(processed_file).name}")

print(f"Input MIDI files found: {total}")
print(f"Extracted melodies: {total}")


# LSTM Model for Melody Continuation

- This section trains a next-token LSTM using the extracted melody MIDI files in `dataset/melodies`.
- The token format follows the reference project style: MIDI pitch / `r` / `_` continuation symbols.


In [ ]:
import numpy as np
import importlib
import tensorflow as tf
import json

# TIME_STEP means each token step represents a quarter of a beat (1/16 note in 4/4).
TIME_STEP: float = 0.25
# SEQUENCE_LENGTH is how many previous tokens the model sees before predicting the next one.
SEQUENCE_LENGTH: int = 64
# BATCH_SIZE is how many training sequences are processed together in one optimizer step.
BATCH_SIZE: int = 128
# EPOCHS is how many full passes over the training dataset we run.
EPOCHS: int = 5
# Path where the trained Keras model will be saved.
MODEL_SAVE_PATH: Path = PROJECT_ROOT / "melody_lstm.keras"
# Path where the token-to-integer lookup table will be saved.
MAPPING_SAVE_PATH: Path = PROJECT_ROOT / "melody_mapping.json"

# Print whether TensorFlow currently detects a GPU in this environment.
print("GPU Status: ", tf.test.is_gpu_available())

def encode_melody_file(midi_path: Path, time_step: float = TIME_STEP) -> list[str]:
    # Parse one MIDI file into a music21 score so we can iterate note/rest events.
    score = converter.parse(str(midi_path))
    # This list stores the tokenized melody for one file.
    tokens: list[str] = []
    # Flatten to one timeline, then iterate over notes and rests in order.
    for event in score.flatten().notesAndRests:
        # Use pitch number for notes (like "60") and "r" for rests.
        symbol = str(int(event.pitch.midi)) if isinstance(event, note.Note) else "r"
        # Convert event duration into number of token steps at the chosen resolution.
        steps = max(1, int(round(float(event.quarterLength) / time_step)))
        # First token marks the event start (note or rest).
        tokens.append(symbol)
        # Remaining time of that event is represented by continuation tokens "_".
        tokens.extend(["_"] * (steps - 1))
    # Return the full token list for this melody file.
    return tokens

def build_split_tokens(split_name: str) -> list[str]:
    # Build path to one dataset split folder (train / validation / test).
    split_dir: Path = MELODY_DIR / split_name
    # Sort for deterministic ordering across runs.
    melody_files = sorted(split_dir.glob("*.midi"))
    # Delimiter tokens separate files so model learns sequence boundaries.
    delimiter = ["/"] * SEQUENCE_LENGTH
    # Accumulate every file's tokens into one long token list.
    tokens = []
    # Encode each melody file and append delimiter after it.
    for f in melody_files:
        tokens.extend(encode_melody_file(f))
        tokens.extend(delimiter)
    # Return combined tokens for this split.
    return tokens

def build_mapping(all_tokens: list[str]) -> dict[str, int]:
    # Create sorted unique vocabulary so token ids are stable and reproducible.
    vocab = sorted(set(all_tokens))
    # Map each token string to a unique integer id.
    return {token: idx for idx, token in enumerate(vocab)}

def prepare_dataset(tokens: list[str], mapping: dict[str, int], seq_len: int, batch_size: int):
    """Convert token strings into a shuffled tf.data pipeline for next-token training."""
    # Replace each token string with its integer id using the saved vocabulary mapping.
    int_tokens = [mapping[t] for t in tokens]
    # Create a TensorFlow dataset from the integer token stream.
    dataset = tf.data.Dataset.from_tensor_slices(int_tokens)

    # Build sliding windows of length seq_len + 1 (inputs + next-token target).
    sequences = dataset.window(seq_len + 1, shift=1, drop_remainder=True)
    # Convert each window dataset into a dense tensor chunk.
    sequences = sequences.flat_map(lambda w: w.batch(seq_len + 1))

    def split_input_target(chunk):
        # Input is first seq_len tokens.
        input_text = chunk[:-1]
        # Target is the immediate next token after the input window.
        target_text = chunk[-1]
        # One-hot encode input ids to shape [seq_len, vocab_size] for the LSTM.
        return tf.one_hot(input_text, depth=len(mapping)), target_text

    # Apply split function to every chunk.
    dataset = sequences.map(split_input_target)
    # Shuffle for better training, then batch, then prefetch for throughput.
    dataset = dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Return the ready-to-train dataset pipeline.
    return dataset

def build_lstm_model(vocab_size: int) -> tf.keras.Model:
    # Stack two LSTM layers plus dropout and a softmax output over vocabulary.
    model = tf.keras.Sequential([
        # Expected input shape: [sequence_length, vocabulary_size] (one-hot vectors).
        tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, vocab_size)),
        # First LSTM returns full sequence so second LSTM can keep processing temporal context.
        tf.keras.layers.LSTM(256, return_sequences=True),
        # Dropout helps reduce overfitting by randomly disabling some activations during training.
        tf.keras.layers.Dropout(0.2),
        # Second LSTM compresses sequence into one context vector.
        tf.keras.layers.LSTM(256),
        # Final dense layer outputs probability for each possible next token.
        tf.keras.layers.Dense(vocab_size, activation="softmax"),
    ])
    # Compile with Adam optimizer and sparse target labels (integer class ids).
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    # Return compiled model ready for training.
    return model

# Build token streams for train and validation splits.
train_tokens = build_split_tokens("train")
validation_tokens = build_split_tokens("validation")
# Combine to build a vocabulary that covers both sets.
all_tokens = train_tokens + validation_tokens
mapping = build_mapping(all_tokens)

# Save mapping so generation code can convert between ids and tokens later.
with MAPPING_SAVE_PATH.open("w") as fp:
    json.dump(mapping, fp)

# Create TensorFlow input pipelines for training and validation.
train_ds = prepare_dataset(train_tokens, mapping, SEQUENCE_LENGTH, BATCH_SIZE)
val_ds = prepare_dataset(validation_tokens, mapping, SEQUENCE_LENGTH, BATCH_SIZE)

# Build model using vocabulary size as output dimension.
model = build_lstm_model(len(mapping))
# Print model architecture summary in notebook output.
model.summary()

# Train model and evaluate on validation set after each epoch.
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
# Save trained model to disk for later generation.
model.save(MODEL_SAVE_PATH)

# Generate from Seed Melody

- This section loads a short seed from a processed melody file, generates continuation tokens, and saves a MIDI file in `results/`.


In [ ]:
RESULTS_DIR: Path = PROJECT_ROOT / "results"
# Create results directory if it does not already exist.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Which split to sample seed melodies from.
SEED_SPLIT: str = "test"
# Number of random seed files to generate from.
NUM_SEED_FILES: int = 5
# How many quarter lengths from each seed file to keep as seed prompt.
SEED_DURATION: float = 8.0
# Maximum number of new tokens to generate after the seed.
GEN_STEPS: int = 256
# Sampling temperature: lower is safer, higher is more diverse.
TEMPERATURE: float = 0.8
# Fixed random seed for reproducible seed-file selection.
RANDOM_SEED: int = 42


def sample_with_temperature(probabilities: np.ndarray, temperature: float) -> int:
    """Sample one token index from probabilities using temperature scaling."""
    # Convert to float64 for stable log/exp operations.
    probs = np.asarray(probabilities, dtype=np.float64)
    # Clamp tiny values to avoid log(0), which is undefined.
    probs = np.maximum(probs, 1e-12)
    # Scale logits by temperature; small temperature sharpens distribution.
    logits = np.log(probs) / max(temperature, 1e-6)
    # Subtract max logit for numerical stability before exponentiation.
    logits -= np.max(logits)
    # Convert stabilized logits back to positive weights.
    dist = np.exp(logits)
    # Normalize so probabilities sum to 1.
    dist /= np.sum(dist)
    # Randomly sample one token id using the adjusted distribution.
    return int(np.random.choice(len(dist), p=dist))


def generate_tokens_from_seed(
    model: tf.keras.Model,
    mapping: dict[str, int],
    seed_tokens: list[str],
    num_steps: int,
    temperature: float,
) -> list[str]:
    """Generate continuation tokens autoregressively from a seed token list."""
    # Reverse lookup from integer id to token string.
    id_to_token = {idx: tok for tok, idx in mapping.items()}
    # Boundary padding so early predictions still have full context length.
    start_symbols = ["/"] * SEQUENCE_LENGTH

    # Keep only tokens known by mapping; remove delimiters from user seed.
    seed = [tok for tok in seed_tokens if tok in mapping and tok != "/"]
    # Generated output starts with the original seed prompt.
    generated = seed.copy()
    # Context is model input history represented as integer ids.
    context = [mapping[tok] for tok in start_symbols + seed]

    # Autoregressive loop: predict one token, append it, repeat.
    for _ in range(num_steps):
        # Use only the last SEQUENCE_LENGTH tokens as model input window.
        context_window = context[-SEQUENCE_LENGTH:]
        # Convert integer ids to one-hot format expected by the model.
        one_hot = tf.keras.utils.to_categorical(
            context_window,
            num_classes=len(mapping),
        ).astype(np.float32)
        # Add batch dimension so shape becomes [1, seq_len, vocab_size].
        one_hot = one_hot[np.newaxis, ...]

        # Predict next-token probability distribution.
        probabilities = model.predict(one_hot, verbose=0)[0]
        # Sample one token id with temperature-controlled randomness.
        next_id = sample_with_temperature(probabilities, temperature)
        # Convert sampled id back into token string.
        next_token = id_to_token[next_id]

        # Append sampled id to context so future predictions can use it.
        context.append(next_id)
        # Stop generation if model predicts delimiter token.
        if next_token == "/":
            break
        # Keep non-delimiter tokens in final generated sequence.
        generated.append(next_token)

    # Return seed + generated continuation tokens.
    return generated


def save_generated_tokens_as_midi(tokens: list[str], output_path: Path, step_duration: float = TIME_STEP) -> None:
    """Convert tokens to music21 stream and write to MIDI."""
    # Create output stream that will hold reconstructed notes and rests.
    out_stream = stream.Stream()

    # start_symbol tracks the current active note/rest token being extended.
    start_symbol: Optional[str] = None
    # step_counter counts how many continuation steps belong to start_symbol.
    step_counter: int = 1

    # Iterate through generated token sequence and merge "_" continuations.
    for i, symbol in enumerate(tokens):
        # If token starts a new event (or this is last token), close previous event first.
        if symbol != "_" or i + 1 == len(tokens):
            # Skip writing on first iteration when there is no previous symbol yet.
            if start_symbol is not None:
                # Convert step count into quarter-length duration.
                ql_duration = step_duration * step_counter
                # Rebuild either a rest or note based on start symbol.
                if start_symbol == "r":
                    event = note.Rest(quarterLength=ql_duration)
                else:
                    event = note.Note(int(start_symbol), quarterLength=ql_duration)
                # Append reconstructed event to output stream in sequence order.
                out_stream.append(event)
                # Reset counter for the next event.
                step_counter = 1
            # Start tracking the new event symbol.
            start_symbol = symbol
        else:
            # Continuation token extends current event by one more time step.
            step_counter += 1

    # Ensure destination directory exists before writing MIDI file.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # Export stream to MIDI on disk.
    out_stream.write("midi", fp=str(output_path))


# Collect seed candidates from selected split and keep deterministic ordering.
seed_files: list[Path] = sorted((MELODY_DIR / SEED_SPLIT).glob("*.midi"))
# Fail early with a clear message if no seed files are available.
if not seed_files:
    raise ValueError(f"No seed files found in split: {SEED_SPLIT}")

# Convert seed duration in quarter lengths into token count.
seed_token_length: int = max(1, int(round(SEED_DURATION / TIME_STEP)))
# Create RNG object with fixed seed for reproducibility.
rng = np.random.default_rng(RANDOM_SEED)
# Do not request more seed files than actually available.
n_selected: int = min(NUM_SEED_FILES, len(seed_files))
# Randomly choose unique seed files to generate from.
selected_seed_files = list(rng.choice(seed_files, size=n_selected, replace=False))

# Load token mapping created during training.
with MAPPING_SAVE_PATH.open("r", encoding="utf-8") as fp:
    saved_mapping: dict[str, int] = json.load(fp)

# Load trained model from disk for inference.
generation_model = tf.keras.models.load_model(MODEL_SAVE_PATH)

# Generate one continuation MIDI for each selected seed file.
for seed_midi_path in selected_seed_files:
    # Encode full seed file into token sequence.
    seed_tokens_all: list[str] = encode_melody_file(seed_midi_path)
    # Keep only initial seed_token_length tokens as prompt.
    seed_tokens: list[str] = seed_tokens_all[:seed_token_length]

    # Run autoregressive token generation from the chosen prompt.
    generated_tokens = generate_tokens_from_seed(
        model=generation_model,
        mapping=saved_mapping,
        seed_tokens=seed_tokens,
        num_steps=GEN_STEPS,
        temperature=TEMPERATURE,
    )

    # Build output filename based on source seed name.
    result_path: Path = RESULTS_DIR / f"{seed_midi_path.stem}_generated.midi"
    # Convert generated tokens back to MIDI and save.
    save_generated_tokens_as_midi(generated_tokens, result_path)

    # Print progress details for this generated file.
    print(f"Seed file: {seed_midi_path.name}")
    print(f"Seed token length: {len(seed_tokens)}")
    print(f"Generated token length: {len(generated_tokens)}")
    print(f"Saved generated MIDI: {result_path}")
